# Optimization Check

This notebook validates the phase-04 optimization flow with persisted reusable precompute artifacts plus batched `K` solves that reuse one built model and warm-start the next budget.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import json
import time

import numpy as np

from src.common.floorplan import FloorPlanInput, NULL_CELL, OPEN_CELL, SOLID_CELL
from src.planner.main import load_workspace
from src.planner.phase01_candidate_generation import generate_candidate_generation_artifacts, resolve_candidate_generation_artifacts
from src.planner.phase03_scoring import SparseScoreArtifacts, decode_configuration_ordinal, resolve_sparse_score_artifacts
from src.planner.phase04_optimization import (
    PHASE_ARTIFACT_STEM,
    PHASE_NAME,
    PHASE_PRECOMPUTE_ARTIFACT_STEM,
    build_optimization_precompute_artifacts,
    load_optimization_artifacts,
    resolve_optimization_artifacts_for_k_values,
    resolve_optimization_precompute_artifacts,
    solve_for_k_values,
    solve_optimization_artifacts,
    validate_optimization_artifacts,
)


In [ ]:
workspace = load_workspace()
phase01_artifacts = resolve_candidate_generation_artifacts(workspace.floorplan, workspace.config, repo_root=workspace.repo_root)
phase03_artifacts = resolve_sparse_score_artifacts(workspace.floorplan, workspace.config, repo_root=workspace.repo_root)
phase04_precompute_artifacts = resolve_optimization_precompute_artifacts(workspace.floorplan, workspace.config, repo_root=workspace.repo_root)
phase04_precompute_path = workspace.artifact_dir / f"{PHASE_PRECOMPUTE_ARTIFACT_STEM}.npz"
threshold_sizes = [int(offsets[-1]) for offsets in phase04_precompute_artifacts.level_offsets]
{
    "phase_name": PHASE_NAME,
    "artifact_dir": str(workspace.artifact_dir),
    "precompute_path": str(phase04_precompute_path),
    "configuration_count": len(phase03_artifacts.configuration_candidate_ordinals),
    "positive_pair_count": int(len(phase03_artifacts.score_target_ordinals)),
    "threshold_level_sizes": threshold_sizes,
}


In [ ]:
def build_synthetic_floorplan() -> FloorPlanInput:
    grid = np.asarray([[NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL], [NULL_CELL, OPEN_CELL, OPEN_CELL, NULL_CELL], [NULL_CELL, OPEN_CELL, OPEN_CELL, NULL_CELL], [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL]], dtype=np.int8)
    return FloorPlanInput(name="phase04-open-square", source_path=Path("phase04-open-square.png"), grid=grid, height=4, width=4, null_cell_count=int(np.count_nonzero(grid == NULL_CELL)), open_cell_count=int(np.count_nonzero(grid == OPEN_CELL)), solid_cell_count=int(np.count_nonzero(grid == SOLID_CELL)), grid_cell_size_m=1.0)

synthetic_floorplan = build_synthetic_floorplan()
synthetic_config = replace(workspace.config, floorplan_name="phase04-open-square", orientation_step_deg=180, k_values=(1, 2))
synthetic_phase01 = generate_candidate_generation_artifacts(synthetic_floorplan, synthetic_config)
synthetic_phase03 = SparseScoreArtifacts(
    grid_shape=synthetic_floorplan.shape,
    candidate_count=4,
    open_cell_count=4,
    orientation_angles_deg=np.array([0.0, 180.0], dtype=np.float32),
    configuration_candidate_ordinals=np.repeat(np.arange(4, dtype=np.int32), 2),
    configuration_angle_ordinals=np.tile(np.arange(2, dtype=np.int16), 4),
    score_configuration_offsets=np.array([0, 2, 4, 6, 6, 7, 8, 8, 8], dtype=np.int32),
    score_target_ordinals=np.array([0, 1, 2, 3, 0, 2, 3, 0], dtype=np.int32),
    score_values=np.array([4, 4, 2, 2, 3, 2, 1, 2], dtype=np.int8),
)
synthetic_precompute = build_optimization_precompute_artifacts(synthetic_floorplan, synthetic_phase01, synthetic_phase03)
synthetic_k1 = solve_optimization_artifacts(synthetic_floorplan, synthetic_config, synthetic_phase01, synthetic_phase03, k=1, precompute_artifacts=synthetic_precompute)
synthetic_batch = solve_for_k_values(synthetic_floorplan, synthetic_config, synthetic_phase01, synthetic_phase03, k_values=(1, 2), precompute_artifacts=synthetic_precompute)
{
    "single_k1_selected": synthetic_k1.selected_configuration_ordinals.tolist(),
    "single_k1_objective": synthetic_k1.objective_value,
    "batch_k_values": [artifact.solved_k for artifact in synthetic_batch],
    "batch_objectives": [artifact.objective_value for artifact in synthetic_batch],
    "batch_last_selected": synthetic_batch[-1].selected_configuration_ordinals.tolist(),
    "batch_last_scores": synthetic_batch[-1].final_open_cell_scores.tolist(),
}


In [ ]:
batch_k_values = tuple(workspace.config.k_values[:2])
start = time.perf_counter()
batch_artifacts = resolve_optimization_artifacts_for_k_values(workspace.floorplan, workspace.config, repo_root=workspace.repo_root, k_values=batch_k_values)
solve_seconds = time.perf_counter() - start
for artifact in batch_artifacts:
    reloaded_artifact = load_optimization_artifacts(workspace.artifact_dir / f"{PHASE_ARTIFACT_STEM}_k{artifact.solved_k}.npz")
    validate_optimization_artifacts(workspace.floorplan, workspace.config, phase01_artifacts, phase03_artifacts, reloaded_artifact)
last_artifact = batch_artifacts[-1]
last_summary_path = workspace.artifact_dir / f"{PHASE_ARTIFACT_STEM}_k{last_artifact.solved_k}_summary.json"
summary_payload = json.loads(last_summary_path.read_text(encoding="utf-8"))
decoded_selected = [decode_configuration_ordinal(phase03_artifacts, int(configuration_ordinal)) for configuration_ordinal in last_artifact.selected_configuration_ordinals[: min(10, len(last_artifact.selected_configuration_ordinals))]]
{
    "solve_k_values": batch_k_values,
    "solve_seconds": solve_seconds,
    "selected_camera_counts": [len(artifact.selected_configuration_ordinals) for artifact in batch_artifacts],
    "objective_values": [artifact.objective_value for artifact in batch_artifacts],
    "last_summary": summary_payload,
    "decoded_selected_sample": decoded_selected,
}
